<a href="https://colab.research.google.com/github/TAlkam/NRD_2017/blob/main/NRD_2017_AD_G30.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NRD 2017 Alzheimer’s Disease Revision Analysis — Clean G30.x Cohort

**Purpose:** rerun the full paper from a single, explicitly defined Alzheimer’s disease cohort and eliminate `df`/`dat`/`d` contamination.

Run this notebook **top to bottom in a fresh Colab runtime**. Do not mix it with cells from the previous notebook.

### Cohort definition
A hospitalization is included when:
1. age is **≥60 years**, and
2. **G30.x** appears in any ICD-10-CM diagnosis field (`i10_dx*`).

This is a **G30-defined Alzheimer’s disease cohort**. A hospitalization is *not* excluded merely because another dementia code (for example F01/F02/F03) is also present.

### Outputs
- Main Table 1
- Main Table 2
- Figures 1–4
- Supplementary Tables S1–S7
- Supplementary Figure S1 (decision curve)
- Cohort audit and software-version files

### Revision v3: binary ED involvement
This version collapses `HCUP_ED` to `ed_involvement` (0 = no evidence of ED services; 1 = any evidence, original HCUP_ED 1–4). The original multi-category `HCUP_ED` is not used as a model predictor, eliminating the sparse category-3 separation issue.


In [ ]:
# CELL 1 — Install packages
!pip -q install pyreadstat xgboost shap scikit-learn statsmodels patsy scipy

In [ ]:
# CELL 2 — Upload the Stata file and load it WITHOUT converting value labels to categories
# Recommended input: your Stata-created G30.x file, e.g. NRD_2017_AD_only.dta

from google.colab import files
import os, re, json, warnings, zipfile, platform, sys
from pathlib import Path

import numpy as np
import pandas as pd

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload exactly one .dta file for this run.")

input_name = next(iter(uploaded.keys()))
if not input_name.lower().endswith(".dta"):
    raise ValueError("This notebook expects a Stata .dta file.")

raw_df = pd.read_stata(input_name, convert_categoricals=False)
raw_df.columns = [str(c).strip().lower() for c in raw_df.columns]

print("Loaded:", input_name)
print("Raw shape:", raw_df.shape)

In [ ]:
# CELL 3 — Create the eligible G30.x cohort (age >=60)
# Cell 4 will exclude records with unknown in-hospital mortality and then LOCK the final analytic cohort.

# Diagnosis columns
DX_COLS = sorted(
    [c for c in raw_df.columns if re.fullmatch(r"i10_dx\d+", c)],
    key=lambda x: int(re.search(r"\d+$", x).group())
)
if not DX_COLS:
    raise ValueError("No i10_dx* diagnosis columns were found.")

required_core = ["age", "died", "los", "discwt"]
missing_core = [c for c in required_core if c not in raw_df.columns]
if missing_core:
    raise ValueError(f"Missing required columns: {missing_core}")


def any_icd_prefix(data, cols, prefixes):
    prefixes = tuple(str(p).upper() for p in prefixes)
    hit = np.zeros(len(data), dtype=bool)
    for c in cols:
        s = data[c].astype("string").str.upper().str.strip()
        hit |= s.str.startswith(prefixes, na=False).to_numpy()
    return pd.Series(hit, index=data.index)

# Independent G30.x verification from diagnosis fields
raw_df["has_g30"] = any_icd_prefix(raw_df, DX_COLS, ["G30"])
raw_df["has_other_dementia_code"] = any_icd_prefix(raw_df, DX_COLS, ["F01", "F02", "F03", "G31"])
raw_df["age"] = pd.to_numeric(raw_df["age"], errors="coerce")

cohort_audit = pd.DataFrame({
    "item": [
        "Rows loaded",
        "Rows with G30.x in any diagnosis field",
        "Rows with G30.x and age >=60",
        "G30.x age>=60 rows also carrying F01/F02/F03/G31"
    ],
    "n": [
        len(raw_df),
        int(raw_df["has_g30"].sum()),
        int((raw_df["has_g30"] & raw_df["age"].ge(60)).sum()),
        int((raw_df["has_g30"] & raw_df["age"].ge(60) & raw_df["has_other_dementia_code"]).sum())
    ]
})

display(cohort_audit)

analysis_df = raw_df.loc[raw_df["has_g30"] & raw_df["age"].ge(60)].copy().reset_index(drop=True)

if len(analysis_df) == 0:
    raise ValueError("The G30.x age>=60 cohort is empty. Check the input file and diagnosis columns.")
if not analysis_df["has_g30"].all():
    raise AssertionError("Non-G30 records entered the analytic cohort.")
if not analysis_df["age"].ge(60).all():
    raise AssertionError("Age <60 records entered the analytic cohort.")

# If Stata provided a flag, verify it rather than relying on it.
for flag in ["has_alzheimers", "ad_only", "ad_g30"]:
    if flag in analysis_df.columns:
        observed = pd.to_numeric(analysis_df[flag], errors="coerce")
        bad = int((observed.fillna(0) != 1).sum())
        print(f"Stata flag `{flag}`: {bad} analytic rows are not coded 1 (G30 diagnosis fields remain source of truth).")

# Prevent accidental use of the full input data later.
del raw_df

print("\nEligible G30.x age>=60 cohort before outcome-completeness exclusion N =", len(analysis_df))
print("Cell 4 will exclude records with missing DIED and lock the final analytic cohort.")
print("No dataframe named df/dat/d will be used anywhere in this notebook.")

In [ ]:
# CELL 4 — Derive outcomes, clinical flags, numeric fields, feature sets, and output folder

from scipy.stats import ttest_ind, mannwhitneyu, chi2, chi2_contingency

OUT = Path("/content/nrd2017_ad_revision_outputs")
OUT.mkdir(parents=True, exist_ok=True)

# Required columns for the paper
required = ["died", "los", "discwt", "age", "i10_ndx", "i10_npr", "totchg"]
missing = [c for c in required if c not in analysis_df.columns]
if missing:
    raise ValueError(f"Missing paper variables: {missing}")

# Outcome / continuous variables
for c in ["died", "los", "discwt", "age", "i10_ndx", "i10_npr", "totchg"]:
    analysis_df[c] = pd.to_numeric(analysis_df[c], errors="coerce")

# Mortality is the study outcome. Records with missing DIED have unknown outcome status
# and must NOT be recoded as survivors. Exclude them from the outcome-complete cohort.
n_eligible = len(analysis_df)
n_missing_died = int(analysis_df["died"].isna().sum())
print(f"Eligible G30.x age>=60 records before DIED exclusion: {n_eligible:,}")
print(f"Excluded because DIED is missing: {n_missing_died:,}")

if n_missing_died > 0:
    analysis_df = analysis_df.loc[analysis_df["died"].notna()].copy().reset_index(drop=True)

analysis_df["died"] = analysis_df["died"].astype(int)
if not set(analysis_df["died"].unique()).issubset({0, 1}):
    raise ValueError(f"DIED contains values other than 0/1: {sorted(analysis_df['died'].unique().tolist())}")

# Add outcome-completeness information to the cohort audit and save it.
outcome_audit = pd.DataFrame({
    "item": [
        "Excluded from mortality analyses because DIED was missing",
        "Final G30.x age>=60 outcome-complete analytic cohort"
    ],
    "n": [n_missing_died, len(analysis_df)]
})
cohort_audit = pd.concat([cohort_audit, outcome_audit], ignore_index=True)
cohort_audit.to_csv(OUT / "Cohort_Audit.csv", index=False)

print(f"FINAL LOCKED analytic cohort N = {len(analysis_df):,}")
print("All subsequent analyses use this outcome-complete `analysis_df` only.")

analysis_df["totchg_10k"] = analysis_df["totchg"] / 10000.0

# Clinical diagnosis-family indicators: presence anywhere in the discharge diagnosis list.
FLAG_MAP = {
    "sepsis": ["A40", "A41"],
    "pneumonia": ["J12", "J13", "J14", "J15", "J16", "J17", "J18"],
    "uti": ["N39"],
    "aki": ["N17"],
    "chf": ["I50"],
    "copd": ["J44"],
    "ckd": ["N18"],
    "stroke": ["I60", "I61", "I62", "I63"],
    "delirium": ["F05"],
}
for name, prefixes in FLAG_MAP.items():
    analysis_df[name] = any_icd_prefix(analysis_df, DX_COLS, prefixes).astype(int)

# Binary numeric fields, if present
for c in ["female", "aweekend", "elective"]:
    if c in analysis_df.columns:
        analysis_df[c] = pd.to_numeric(analysis_df[c], errors="coerce")

# HCUP_ED is an administrative indicator of evidence of emergency-department services.
# Collapse the original categories to a clinically interpretable binary indicator:
#   0 = no evidence of ED services (HCUP_ED == 0)
#   1 = any evidence of ED services (HCUP_ED >= 1)
# Missing HCUP_ED remains missing and is handled by the model preprocessing/imputation steps.
if "hcup_ed" not in analysis_df.columns:
    raise ValueError("HCUP_ED is required to derive binary ED involvement.")
analysis_df["hcup_ed"] = pd.to_numeric(analysis_df["hcup_ed"], errors="coerce")
analysis_df["ed_involvement"] = np.where(
    analysis_df["hcup_ed"].isna(),
    np.nan,
    (analysis_df["hcup_ed"] >= 1).astype(int)
)

print("\nHCUP_ED collapsed to binary ED involvement:")
print(pd.crosstab(analysis_df["ed_involvement"], analysis_df["died"], margins=True, dropna=False))

# Explicit categorical features: these are coded numerically in HCUP but must be treated categorically.
# ED involvement is binary and is therefore modeled as a 0/1 numeric variable rather than as HCUP_ED categories.
CATEGORICAL_FEATURES = [c for c in ["pay1", "zipinc_qrtl", "pl_nchs"] if c in analysis_df.columns]
for c in CATEGORICAL_FEATURES:
    analysis_df[c] = pd.to_numeric(analysis_df[c], errors="coerce")

BASE_FEATURES = [c for c in [
    "age", "female",
    "sepsis", "aki", "stroke", "uti", "chf", "ckd", "pneumonia", "delirium", "copd",
    "i10_ndx",
    "aweekend", "elective",
    "pay1", "zipinc_qrtl", "pl_nchs", "ed_involvement"
] if c in analysis_df.columns]

COURSE_FEATURES = [c for c in ["los", "i10_npr", "totchg"] if c in analysis_df.columns]
FEATURES_A = BASE_FEATURES.copy()
FEATURES_B = BASE_FEATURES + [c for c in COURSE_FEATURES if c not in BASE_FEATURES]

TARGET = "died"
GROUP = "nrd_visitlink"
N_SPLITS = 5
RANDOM_STATE = 42

if GROUP not in analysis_df.columns:
    raise ValueError("NRD_VisitLink is required for patient-grouped cross-validation.")

print("Cohort N:", len(analysis_df))
print("Deaths:", int(analysis_df[TARGET].sum()))
print("Unweighted mortality %:", round(100 * analysis_df[TARGET].mean(), 3))
print("Weighted national N:", round(analysis_df["discwt"].sum()))
print("Weighted deaths:", round((analysis_df["discwt"] * analysis_df[TARGET]).sum()))
print("Weighted mortality %:", round(100 * np.average(analysis_df[TARGET], weights=analysis_df["discwt"]), 3))
print("Missing LOS:", int(analysis_df["los"].isna().sum()))
print("Model A features:", FEATURES_A)
print("Model B adds:", [c for c in FEATURES_B if c not in FEATURES_A])

## Main Table 1 — cohort characteristics by in-hospital mortality

In [ ]:
# CELL 5 — Main Table 1
# Weighted descriptive estimates; unweighted survivor-vs-decedent tests.


def wmean(s, w):
    m = s.notna() & w.notna()
    return np.average(s[m], weights=w[m]) if m.any() else np.nan


def weighted_quantile(values, quantiles, weights):
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)
    mask = np.isfinite(values) & np.isfinite(weights) & (weights >= 0)
    values, weights = values[mask], weights[mask]
    if len(values) == 0 or weights.sum() == 0:
        return np.repeat(np.nan, len(quantiles))
    order = np.argsort(values)
    values, weights = values[order], weights[order]
    cdf = np.cumsum(weights) / np.sum(weights)
    return np.interp(quantiles, cdf, values)


def p_ttest(var):
    a = analysis_df.loc[analysis_df[TARGET] == 0, var].dropna().to_numpy(float)
    b = analysis_df.loc[analysis_df[TARGET] == 1, var].dropna().to_numpy(float)
    return ttest_ind(a, b, equal_var=False).pvalue


def p_mwu(var):
    a = analysis_df.loc[analysis_df[TARGET] == 0, var].dropna().to_numpy(float)
    b = analysis_df.loc[analysis_df[TARGET] == 1, var].dropna().to_numpy(float)
    return mannwhitneyu(a, b, alternative="two-sided").pvalue


def fp(p):
    if pd.isna(p): return ""
    return "<0.001" if p < 0.001 else f"{p:.3f}"

surv = analysis_df[analysis_df[TARGET] == 0]
dead = analysis_df[analysis_df[TARGET] == 1]
w = analysis_df["discwt"]

q_surv = weighted_quantile(surv["los"], [0.25, 0.50, 0.75], surv["discwt"])
q_dead = weighted_quantile(dead["los"], [0.25, 0.50, 0.75], dead["discwt"])

TABLE1 = pd.DataFrame([
    ["Hospitalizations, n (unweighted)", f"{len(analysis_df):,}", f"{len(surv):,}", f"{len(dead):,}", "—"],
    ["In-hospital deaths, n (%) (unweighted)", f"{int(analysis_df[TARGET].sum()):,} ({100*analysis_df[TARGET].mean():.2f})", "—", "—", "—"],
    ["National estimate, n (weighted)", f"{analysis_df['discwt'].sum():,.0f}", f"{surv['discwt'].sum():,.0f}", f"{dead['discwt'].sum():,.0f}", "—"],
    ["In-hospital mortality, % (weighted)", f"{100*np.average(analysis_df[TARGET], weights=w):.2f}", "—", "—", "—"],
    ["Age, years (weighted mean)", "—", f"{wmean(surv['age'], surv['discwt']):.2f}", f"{wmean(dead['age'], dead['discwt']):.2f}", fp(p_ttest('age'))],
    ["Number of diagnoses (weighted mean)", "—", f"{wmean(surv['i10_ndx'], surv['discwt']):.2f}", f"{wmean(dead['i10_ndx'], dead['discwt']):.2f}", fp(p_ttest('i10_ndx'))],
    ["Number of procedures (weighted mean)", "—", f"{wmean(surv['i10_npr'], surv['discwt']):.2f}", f"{wmean(dead['i10_npr'], dead['discwt']):.2f}", fp(p_ttest('i10_npr'))],
    ["Length of stay, days (weighted median [IQR])", "—", f"{q_surv[1]:.0f} [{q_surv[0]:.0f}–{q_surv[2]:.0f}]", f"{q_dead[1]:.0f} [{q_dead[0]:.0f}–{q_dead[2]:.0f}]", fp(p_mwu('los'))],
], columns=["Characteristic", "Overall", "Survived", "Died in-hospital", "p-value"])

display(TABLE1)
TABLE1.to_csv(OUT / "Table1_Main_Cohort_Characteristics.csv", index=False)

## Figure 1 and Supplementary Table S1 — unadjusted LOS–mortality relationship

In [ ]:
# CELL 6 — Figure 1 + Supplementary Table S1

import matplotlib.pyplot as plt
from statsmodels.stats.proportion import proportion_confint

los_df = analysis_df.loc[analysis_df["los"].notna() & analysis_df["los"].ge(0), ["los", TARGET]].copy()

bins = [-0.1, 1, 3, 6, 9, 14, 21, np.inf]
labels = ["0–1", "2–3", "4–6", "7–9", "10–14", "15–21", "≥22"]
los_df["los_bin"] = pd.cut(los_df["los"], bins=bins, labels=labels, right=True)

S1 = (los_df.groupby("los_bin", observed=True)[TARGET]
      .agg(n="size", deaths="sum", death_rate="mean")
      .reset_index())

ci = [proportion_confint(int(k), int(n), alpha=0.05, method="wilson") for k, n in zip(S1["deaths"], S1["n"])]
S1["mortality_pct"] = 100 * S1["death_rate"]
S1["ci_low_pct"] = [100*x[0] for x in ci]
S1["ci_high_pct"] = [100*x[1] for x in ci]
S1["percent_of_admissions"] = 100 * S1["n"] / len(los_df)
S1["percent_of_all_deaths"] = 100 * S1["deaths"] / analysis_df[TARGET].sum()

display(S1)
S1.to_csv(OUT / "Supplementary_Table_S1_LOS_Mortality.csv", index=False)

x = np.arange(len(S1))
y = S1["mortality_pct"].to_numpy()
yerr = np.vstack([y - S1["ci_low_pct"].to_numpy(), S1["ci_high_pct"].to_numpy() - y])
xt = [f"{b}\n(n={int(n)})" for b, n in zip(S1["los_bin"].astype(str), S1["n"])]

plt.figure(figsize=(9, 4.6))
plt.errorbar(x, y, yerr=yerr, fmt="o-", capsize=4)
plt.xticks(x, xt)
plt.xlabel("Length of stay (days)")
plt.ylabel("In-hospital death rate (%) with 95% CI")
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.savefig(OUT / "Figure1_LOS_Mortality.png", dpi=600, bbox_inches="tight")
plt.show()

print(f"LOS-binned denominator: {len(los_df):,}")
print(f"Excluded because LOS missing/invalid: {len(analysis_df)-len(los_df):,}")

## Main Table 2 — weighted multivariable regression with restricted cubic spline LOS

In [ ]:
# CELL 7 — Main Table 2, restricted cubic spline test, and variance sensitivity

import statsmodels.api as sm
import statsmodels.formula.api as smf

# Keep one modeling frame; imputation rules mirror the original analytic notebook but are now explicit.
REG_NUMERIC = [c for c in [
    "age", "female", "aweekend", "elective", "ed_involvement", "i10_ndx", "i10_npr", "totchg_10k", "los",
    "sepsis", "aki", "stroke", "uti", "chf", "ckd", "pneumonia", "delirium", "copd"
] if c in analysis_df.columns]
REG_CATEGORICAL = CATEGORICAL_FEATURES.copy()


def prepare_regression_data(data):
    out = data.copy()
    for c in REG_NUMERIC:
        out[c] = pd.to_numeric(out[c], errors="coerce")
        if out[c].isna().any():
            out[c] = out[c].fillna(out[c].median())
    for c in REG_CATEGORICAL:
        out[c] = pd.to_numeric(out[c], errors="coerce")
        if out[c].isna().any():
            mode = out[c].mode(dropna=True)
            if mode.empty:
                raise ValueError(f"Categorical variable {c} is entirely missing.")
            out[c] = out[c].fillna(mode.iloc[0])
    out["discwt"] = pd.to_numeric(out["discwt"], errors="coerce")
    if out["discwt"].isna().any():
        raise ValueError("Missing DISCWT values cannot be used in weighted regression.")
    return out

reg_df = prepare_regression_data(analysis_df)

# Explicit reference groups used in the submitted Table 2.
PREFERRED_REFS = {"pay1": 1, "zipinc_qrtl": 1, "pl_nchs": 1}


def categorical_term(data, var):
    vals = sorted(pd.Series(data[var].dropna().unique()).tolist())
    pref = PREFERRED_REFS.get(var, vals[0])
    ref = pref if pref in vals else vals[0]
    if ref != pref:
        warnings.warn(f"Preferred reference {pref} absent for {var}; using {ref}.")
    return f"C({var}, Treatment(reference={repr(ref)}))", ref

cat_terms = {}
for c in REG_CATEGORICAL:
    cat_terms[c], _ = categorical_term(reg_df, c)

base_formula_terms = [c for c in [
    "age", "female", "sepsis", "aki", "stroke", "i10_ndx", "chf", "uti", "ckd",
    "copd", "delirium", "elective", "pneumonia", "aweekend", "ed_involvement"
] if c in reg_df.columns]
base_formula_terms += [cat_terms[c] for c in REG_CATEGORICAL]

FORMULA_A = f"{TARGET} ~ " + " + ".join(base_formula_terms)
FORMULA_B = FORMULA_A + " + i10_npr + totchg_10k + cr(los, df=5, constraints='center')"
FORMULA_B_LINEAR = FORMULA_A + " + i10_npr + totchg_10k + los"


def fit_weighted_formula(formula, data, cov_type="HC1", cluster=None):
    model = smf.glm(formula=formula, data=data, family=sm.families.Binomial(), freq_weights=data["discwt"])
    if cov_type == "cluster":
        if cluster is None:
            raise ValueError("cluster groups required")
        return model.fit(cov_type="cluster", cov_kwds={"groups": cluster})
    return model.fit(cov_type=cov_type)

resA = fit_weighted_formula(FORMULA_A, reg_df, cov_type="HC1")
resB = fit_weighted_formula(FORMULA_B, reg_df, cov_type="HC1")
resB_linear = fit_weighted_formula(FORMULA_B_LINEAR, reg_df, cov_type="HC1")

# Overall LOS spline term Wald test
wt = resB.wald_test_terms(skip_single=False)
los_term = [idx for idx in wt.table.index if str(idx).startswith("cr(los")][0]
LOS_SPLINE_WALD_P = float(wt.table.loc[los_term, "pvalue"])

# RCS vs linear LOS likelihood-ratio comparison (secondary formal nonlinearity check)
lr_stat = 2 * (resB.llf - resB_linear.llf)
lr_df = max(int(round(resB.df_model - resB_linear.df_model)), 1)
LOS_NONLINEAR_LRT_P = float(chi2.sf(lr_stat, lr_df))

print("Formula A:", FORMULA_A)
print("Formula B:", FORMULA_B)
print(f"Overall LOS spline Wald p = {LOS_SPLINE_WALD_P:.3e}")
print(f"RCS-vs-linear LOS likelihood-ratio p = {LOS_NONLINEAR_LRT_P:.3e}")

# Coefficient label helpers
BASE_LABELS = {
    "age": "Age (per year)", "female": "Female sex (ref: male)",
    "sepsis": "Sepsis", "aki": "Acute kidney injury", "stroke": "Stroke",
    "i10_ndx": "Number of diagnoses (I10_NDX)", "chf": "Congestive heart failure",
    "uti": "Urinary tract infection", "ckd": "Chronic kidney disease",
    "copd": "COPD", "delirium": "Delirium", "elective": "Elective admission (ref: non-elective)",
    "pneumonia": "Pneumonia", "aweekend": "Weekend admission (ref: weekday)",
    "ed_involvement": "ED involvement (any evidence vs none)",
    "i10_npr": "Number of procedures (I10_NPR)", "totchg_10k": "Total charges (per $10,000)"
}
CAT_LABELS = {
    "pay1": {1:"Medicare",2:"Medicaid",3:"Private insurance",4:"Self-pay",5:"No charge",6:"Other"},
    "zipinc_qrtl": {1:"quartile 1",2:"quartile 2",3:"quartile 3",4:"quartile 4"},
    "pl_nchs": {1:"category 1",2:"category 2",3:"category 3",4:"category 4",5:"category 5",6:"category 6"},
}


def normalize_level(x):
    try:
        f = float(x)
        return int(f) if f.is_integer() else f
    except Exception:
        return x


def coef_label(term):
    if term in BASE_LABELS:
        return BASE_LABELS[term]
    m = re.match(r"C\((\w+), Treatment\(reference=.*?\)\)\[T\.(.*?)\]$", term)
    if m:
        var, lev_raw = m.groups()
        lev = normalize_level(lev_raw)
        ref = PREFERRED_REFS.get(var, None)
        lev_name = CAT_LABELS.get(var, {}).get(lev, str(lev))
        ref_name = CAT_LABELS.get(var, {}).get(ref, str(ref))
        if var == "pay1": return f"Primary payer: {lev_name} (vs {ref_name})"
        if var == "zipinc_qrtl": return f"ZIP income {lev_name} (vs {ref_name})"
        if var == "pl_nchs": return f"Urban–rural {lev_name} (vs {ref_name})"
    return term


def result_table(res):
    out = pd.DataFrame({
        "term": res.params.index,
        "OR": np.exp(res.params.values),
        "CI_low": np.exp(res.params.values - 1.96*res.bse.values),
        "CI_high": np.exp(res.params.values + 1.96*res.bse.values),
        "p": res.pvalues.values
    })
    out = out[~out["term"].eq("Intercept")]
    out = out[~out["term"].str.startswith("cr(los", na=False)]
    return out.set_index("term")

TA = result_table(resA)
TB = result_table(resB)
terms = list(dict.fromkeys(list(TA.index) + list(TB.index)))


def fmt_or(row):
    return f"{row.OR:.2f} ({row.CI_low:.2f}–{row.CI_high:.2f})"

def fmtpval(x):
    return "<0.001" if x < 0.001 else f"{x:.3f}"

rows = []
for term in terms:
    a = TA.loc[term] if term in TA.index else None
    b = TB.loc[term] if term in TB.index else None
    rows.append({
        "Predictor": coef_label(term),
        "Model A OR (95% CI)": "" if a is None else fmt_or(a),
        "p (A)": "" if a is None else fmtpval(a.p),
        "Model B OR (95% CI)": "" if b is None else fmt_or(b),
        "p (B)": "" if b is None else fmtpval(b.p),
    })
TABLE2 = pd.DataFrame(rows)

display(TABLE2)
TABLE2.to_csv(OUT / "Table2_Weighted_Logistic_ModelA_vs_ModelB.csv", index=False)

pd.DataFrame({
    "test": ["Overall LOS spline term Wald test", "RCS vs linear LOS likelihood-ratio test"],
    "p_value": [LOS_SPLINE_WALD_P, LOS_NONLINEAR_LRT_P]
}).to_csv(OUT / "Table2_LOS_Spline_Tests.csv", index=False)

# Sensitivity: hospital-clustered covariance, if HOSP_NRD is available.
if "hosp_nrd" in reg_df.columns:
    resA_cluster = fit_weighted_formula(FORMULA_A, reg_df, cov_type="cluster", cluster=reg_df["hosp_nrd"])
    resB_cluster = fit_weighted_formula(FORMULA_B, reg_df, cov_type="cluster", cluster=reg_df["hosp_nrd"])
    sens = pd.DataFrame({
        "term": resA.params.index.union(resB.params.index),
    })
    sens["A_HC1_p"] = sens["term"].map(resA.pvalues)
    sens["A_cluster_p"] = sens["term"].map(resA_cluster.pvalues)
    sens["B_HC1_p"] = sens["term"].map(resB.pvalues)
    sens["B_cluster_p"] = sens["term"].map(resB_cluster.pvalues)
    sens.to_csv(OUT / "Regression_Variance_Sensitivity_HC1_vs_HospitalCluster.csv", index=False)
    print("Saved hospital-cluster covariance sensitivity.")
else:
    print("HOSP_NRD not present; hospital-cluster covariance sensitivity skipped.")

## Supplementary Table S2 — spline flexibility sensitivity (df 4–6) with patient-grouped out-of-fold evaluation

In [ ]:
# CELL 8 — Supplementary Table S2

from patsy import dmatrix, build_design_matrices
from sklearn.model_selection import GroupKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, log_loss


def make_logit_pipeline(feature_names):
    cat = [c for c in feature_names if c in CATEGORICAL_FEATURES]
    num = [c for c in feature_names if c not in cat]
    pre = ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                          ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat),
    ], remainder="drop")
    model = LogisticRegression(penalty="l2", solver="saga", max_iter=10000, random_state=RANDOM_STATE)
    return Pipeline([("pre", pre), ("model", model)])

s2_df = analysis_df.dropna(subset=[TARGET, GROUP, "los"]).copy().reset_index(drop=True)
X0 = s2_df[FEATURES_B].copy()
y0 = s2_df[TARGET].astype(int).to_numpy()
g0 = s2_df[GROUP].astype(str).to_numpy()
gkf = GroupKFold(n_splits=N_SPLITS)

S2_rows = []
for spline_df in [4, 5, 6]:
    oof = np.full(len(s2_df), np.nan)
    fold_aucs = []
    for fold, (tr, te) in enumerate(gkf.split(X0, y0, groups=g0), 1):
        Xtr = X0.iloc[tr].copy(); Xte = X0.iloc[te].copy()
        ytr = y0[tr]; yte = y0[te]

        # Fit natural/restricted cubic spline basis on TRAINING LOS only, then apply same basis to validation.
        tr_spline = dmatrix("cr(los, df=%d, constraints='center') - 1" % spline_df,
                            Xtr, return_type="dataframe")
        design_info = tr_spline.design_info
        te_spline = build_design_matrices([design_info], Xte, return_type="dataframe")[0]
        tr_spline.columns = [f"los_rcs_{j}" for j in range(tr_spline.shape[1])]
        te_spline.columns = tr_spline.columns

        base_no_los = [c for c in FEATURES_B if c != "los"]
        Xtr_aug = pd.concat([Xtr[base_no_los].reset_index(drop=True), tr_spline.reset_index(drop=True)], axis=1)
        Xte_aug = pd.concat([Xte[base_no_los].reset_index(drop=True), te_spline.reset_index(drop=True)], axis=1)

        pipe = make_logit_pipeline(list(Xtr_aug.columns))
        pipe.fit(Xtr_aug, ytr)
        p = pipe.predict_proba(Xte_aug)[:, 1]
        oof[te] = p
        fold_aucs.append(roc_auc_score(yte, p))

    if np.isnan(oof).any():
        raise AssertionError(f"OOF NaNs for spline df={spline_df}")
    S2_rows.append({
        "spline_df": spline_df,
        "pooled_OOF_AUROC": roc_auc_score(y0, oof),
        "fold_AUROC_mean": np.mean(fold_aucs),
        "fold_AUROC_SD": np.std(fold_aucs, ddof=1),
        "pooled_OOF_AUPRC": average_precision_score(y0, oof),
        "pooled_OOF_log_loss": log_loss(y0, np.clip(oof, 1e-12, 1-1e-12))
    })

S2 = pd.DataFrame(S2_rows)
display(S2)
S2.to_csv(OUT / "Supplementary_Table_S2_Spline_DF_Sensitivity.csv", index=False)

## Grouped XGBoost evaluation — one primary evaluation used everywhere

In [ ]:
# CELL 9 — Shared XGBoost pipeline and patient-grouped OOF predictions for Models A and B
# IMPORTANT: no ordinary StratifiedKFold appears anywhere in this notebook.

from sklearn.metrics import (
    roc_auc_score, average_precision_score, roc_curve,
    precision_recall_curve, precision_score, recall_score, f1_score, confusion_matrix
)
from xgboost import XGBClassifier
import shap


def make_xgb_pipeline(features):
    cat = [c for c in features if c in CATEGORICAL_FEATURES]
    num = [c for c in features if c not in cat]
    pre = ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median"))]), num),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                          ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), cat),
    ], remainder="drop", verbose_feature_names_out=False)

    # No class weighting: this preserves the probability scale needed for DCA.
    model = XGBClassifier(
        n_estimators=600,
        learning_rate=0.03,
        max_depth=4,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_lambda=1.0,
        min_child_weight=1.0,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        tree_method="hist"
    )
    return Pipeline([("pre", pre), ("model", model)])


def grouped_xgb_oof(data, features, label):
    work = data[[TARGET, GROUP] + features].copy()
    work = work.dropna(subset=[TARGET, GROUP]).reset_index(drop=True)
    X = work[features]
    y = work[TARGET].astype(int).to_numpy()
    groups = work[GROUP].astype(str).to_numpy()

    gkf = GroupKFold(n_splits=N_SPLITS)
    oof = np.full(len(work), np.nan)
    fold_rows = []
    fold_models = []

    for fold, (tr, te) in enumerate(gkf.split(X, y, groups=groups), 1):
        Xtr, Xte = X.iloc[tr], X.iloc[te]
        ytr, yte = y[tr], y[te]
        pipe = make_xgb_pipeline(features)
        pipe.fit(Xtr, ytr)
        p = pipe.predict_proba(Xte)[:, 1]
        oof[te] = p
        fold_rows.append({
            "model": label, "fold": fold, "n_test": len(te), "deaths_test": int(yte.sum()),
            "AUROC": roc_auc_score(yte, p),
            "AUPRC": average_precision_score(yte, p)
        })
        fold_models.append((fold, tr, te, pipe))

    if np.isnan(oof).any():
        raise AssertionError(f"OOF predictions contain NaNs for {label}")

    pooled = {
        "model": label,
        "n": len(work),
        "deaths": int(y.sum()),
        "prevalence": y.mean(),
        "pooled_OOF_AUROC": roc_auc_score(y, oof),
        "pooled_OOF_AUPRC": average_precision_score(y, oof),
    }
    folds = pd.DataFrame(fold_rows)
    pooled["fold_AUROC_mean"] = folds["AUROC"].mean()
    pooled["fold_AUROC_SD"] = folds["AUROC"].std(ddof=1)
    pooled["fold_AUPRC_mean"] = folds["AUPRC"].mean()
    pooled["fold_AUPRC_SD"] = folds["AUPRC"].std(ddof=1)

    return {"work": work, "X": X, "y": y, "groups": groups, "oof": oof,
            "pooled": pooled, "folds": folds, "fold_models": fold_models, "features": features}

ML_A = grouped_xgb_oof(analysis_df, FEATURES_A, "Model A (admission-only)")
ML_B = grouped_xgb_oof(analysis_df, FEATURES_B, "Model B (inpatient-course)")

# Both models must be evaluated on the same patient-grouped sample.
assert len(ML_A["y"]) == len(ML_B["y"])
assert np.array_equal(ML_A["y"], ML_B["y"])

ML_PERFORMANCE = pd.DataFrame([ML_A["pooled"], ML_B["pooled"]])
display(ML_PERFORMANCE)
ML_PERFORMANCE.to_csv(OUT / "Primary_XGBoost_GroupKFold_OOF_Performance.csv", index=False)
pd.concat([ML_A["folds"], ML_B["folds"]], ignore_index=True).to_csv(OUT / "Primary_XGBoost_Fold_Performance.csv", index=False)

## Figure 2 and Supplementary Table S3 — pooled patient-grouped OOF performance

In [ ]:
# CELL 10 — Figure 2 + Supplementary Table S3

# Figure 2: pooled OOF ROC only. This is the same metric reported in the Abstract and Results.
y = ML_A["y"]
pA = ML_A["oof"]
pB = ML_B["oof"]
aucA = ML_A["pooled"]["pooled_OOF_AUROC"]
aucB = ML_B["pooled"]["pooled_OOF_AUROC"]

fprA, tprA, _ = roc_curve(y, pA)
fprB, tprB, _ = roc_curve(y, pB)

plt.figure(figsize=(6.6, 5.6))
plt.plot(fprA, tprA, label=f"Model A: AUROC {aucA:.3f}")
plt.plot(fprB, tprB, label=f"Model B: AUROC {aucB:.3f}")
plt.plot([0, 1], [0, 1], "--", linewidth=1)
plt.xlabel("1 − Specificity (False Positive Rate)")
plt.ylabel("Sensitivity (True Positive Rate)")
plt.title("Patient-grouped 5-fold out-of-fold ROC")
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig(OUT / "Figure2_GroupKFold_Pooled_OOF_ROC.png", dpi=600, bbox_inches="tight")
plt.show()


def threshold_rows(y, p, model_label):
    prec_curve, rec_curve, thr_curve = precision_recall_curve(y, p)
    f1_curve = 2 * prec_curve * rec_curve / (prec_curve + rec_curve + 1e-12)
    best_i = int(np.nanargmax(f1_curve[:-1]))
    best_thr = float(thr_curve[best_i])
    thresholds = [("best_F1", best_thr), ("0.05", 0.05), ("0.10", 0.10), ("0.15", 0.15), ("0.20", 0.20), ("0.50", 0.50)]
    rows = []
    for name, t in thresholds:
        yh = (p >= t).astype(int)
        tn, fp_, fn, tp = confusion_matrix(y, yh, labels=[0,1]).ravel()
        rows.append({
            "model": model_label, "threshold_label": name, "threshold": t,
            "precision": precision_score(y, yh, zero_division=0),
            "recall": recall_score(y, yh, zero_division=0),
            "F1": f1_score(y, yh, zero_division=0),
            "flagged_pct": 100*yh.mean(), "TN": tn, "FP": fp_, "FN": fn, "TP": tp
        })
    return rows

S3 = pd.DataFrame(threshold_rows(y, pA, "Model A") + threshold_rows(y, pB, "Model B"))
display(S3)
S3.to_csv(OUT / "Supplementary_Table_S3_Threshold_Metrics.csv", index=False)

## Figure 3 and Supplementary Table S4 — grouped out-of-fold SHAP with common feature ordering

In [ ]:
# CELL 11 — Figure 3A/3B + Supplementary Table S4
# SHAP is calculated only on each held-out fold and aggregated to ORIGINAL variables.
# One-hot category contributions are summed back to their parent variable.

FEATURE_LABELS = {
    "age":"Age (years)", "female":"Female sex", "sepsis":"Sepsis",
    "aki":"Acute kidney injury", "stroke":"Stroke", "uti":"Urinary tract infection",
    "chf":"Congestive heart failure", "ckd":"Chronic kidney disease",
    "pneumonia":"Pneumonia", "delirium":"Delirium",
    "copd":"Chronic obstructive pulmonary disease", "i10_ndx":"Number of diagnoses (ICD-10-CM)",
    "aweekend":"Weekend admission", "elective":"Elective admission", "pay1":"Primary payer",
    "zipinc_qrtl":"ZIP income quartile", "pl_nchs":"Urban–rural category",
    "ed_involvement":"Emergency department involvement", "los":"Length of stay (days)",
    "i10_npr":"Number of procedures", "totchg":"Total hospital charges"
}


def parent_feature(transformed_name, features):
    if transformed_name in features:
        return transformed_name
    # OHE names use parent_feature + '_' + category when verbose_feature_names_out=False.
    candidates = [f for f in features if transformed_name.startswith(f + "_")]
    if candidates:
        return max(candidates, key=len)
    raise ValueError(f"Cannot map transformed feature `{transformed_name}` to an original feature.")


def grouped_shap(ml_result):
    X = ml_result["X"]
    features = ml_result["features"]
    shap_all = np.full((len(X), len(features)), np.nan)
    value_all = X[features].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=float)
    stability_rows = []

    for fold, tr, te, pipe in ml_result["fold_models"]:
        Xte = X.iloc[te]
        pre = pipe.named_steps["pre"]
        model = pipe.named_steps["model"]
        Xt = pre.transform(Xte)
        names = list(pre.get_feature_names_out())

        explainer = shap.TreeExplainer(model)
        sv = explainer.shap_values(Xt)
        if isinstance(sv, list):
            sv = sv[-1]
        sv = np.asarray(sv)

        # Aggregate transformed columns back to original variables.
        agg = np.zeros((len(te), len(features)))
        for j, name in enumerate(names):
            parent = parent_feature(str(name), features)
            agg[:, features.index(parent)] += sv[:, j]
        shap_all[te, :] = agg

        fold_imp = pd.Series(np.abs(agg).mean(axis=0), index=features).sort_values(ascending=False)
        for rank, feat in enumerate(fold_imp.head(10).index, 1):
            stability_rows.append({"fold": fold, "feature": feat, "rank": rank,
                                   "mean_abs_SHAP_fold": fold_imp[feat]})

    if np.isnan(shap_all).any():
        raise AssertionError("SHAP matrix contains unfilled rows.")

    mean_abs = pd.Series(np.abs(shap_all).mean(axis=0), index=features).sort_values(ascending=False)
    stab_raw = pd.DataFrame(stability_rows)
    stab = (stab_raw.groupby("feature")
            .agg(appear_top10_folds=("fold", "nunique"),
                 avg_rank_when_in_top10=("rank", "mean"),
                 mean_abs_SHAP_across_top10_folds=("mean_abs_SHAP_fold", "mean"))
            .reset_index()
            .sort_values(["appear_top10_folds", "avg_rank_when_in_top10"], ascending=[False, True]))
    return {"shap": shap_all, "values": value_all, "mean_abs": mean_abs, "stability": stab, "features": features}

SHAP_A = grouped_shap(ML_A)
SHAP_B = grouped_shap(ML_B)

# Common parameters appear in exactly the same order in both panels.
shared = FEATURES_A.copy()
combined_importance = pd.Series({
    f: (SHAP_A["mean_abs"].get(f, 0) + SHAP_B["mean_abs"].get(f, 0))/2 for f in shared
}).sort_values(ascending=False)
COMMON_ORDER = combined_importance.index.tolist()
COURSE_ORDER = [f for f in SHAP_B["mean_abs"].index if f not in shared]
ORDER_A = COMMON_ORDER
ORDER_B = COMMON_ORDER + COURSE_ORDER


def plot_grouped_beeswarm(shap_result, order, title, filename):
    idx = [shap_result["features"].index(f) for f in order]
    sv = shap_result["shap"][:, idx]
    vals = shap_result["values"][:, idx]
    labels = [FEATURE_LABELS.get(f, f) for f in order]
    plt.figure(figsize=(8, max(5, 0.30*len(order)+1.5)))
    shap.summary_plot(sv, vals, feature_names=labels, sort=False, show=False, max_display=len(order))
    plt.title(title)
    plt.tight_layout()
    plt.savefig(OUT / filename, dpi=600, bbox_inches="tight")
    plt.show()

plot_grouped_beeswarm(SHAP_A, ORDER_A, "Model A (admission-only): grouped OOF SHAP", "Figure3A_SHAP_ModelA.png")
plot_grouped_beeswarm(SHAP_B, ORDER_B, "Model B (inpatient-course): grouped OOF SHAP", "Figure3B_SHAP_ModelB.png")

S4A = SHAP_A["stability"].copy(); S4A.insert(0, "model", "Model A")
S4B = SHAP_B["stability"].copy(); S4B.insert(0, "model", "Model B")
S4 = pd.concat([S4A, S4B], ignore_index=True)
S4["feature_label"] = S4["feature"].map(lambda x: FEATURE_LABELS.get(x, x))
display(S4)
S4.to_csv(OUT / "Supplementary_Table_S4_SHAP_Stability.csv", index=False)

## Figure 4 — adjusted nonlinear LOS–mortality association with early-stay sensitivity exclusions

In [ ]:
# CELL 12 — Figure 4
# Uses the full Model B covariate set, not a reduced adjustment set.


def typical_value(data, var):
    if var in REG_CATEGORICAL:
        m = data[var].mode(dropna=True)
        return m.iloc[0]
    return pd.to_numeric(data[var], errors="coerce").median()


def fit_predict_rcs_subset(data, label, spline_df=5):
    sub = prepare_regression_data(data)
    res = fit_weighted_formula(FORMULA_B.replace("df=5", f"df={spline_df}"), sub, cov_type="HC1")

    lo = max(0, float(pd.to_numeric(sub["los"], errors="coerce").quantile(0.01)))
    hi = float(pd.to_numeric(sub["los"], errors="coerce").quantile(0.99))
    grid = np.linspace(lo, hi, 200)

    new = pd.DataFrame({"los": grid})
    needed = set(BASE_FEATURES + ["i10_npr", "totchg_10k"])
    for var in needed:
        if var in sub.columns:
            new[var] = typical_value(sub, var)

    pred = res.get_prediction(new).summary_frame(alpha=0.05)
    return pd.DataFrame({
        "LOS": grid,
        "predicted_probability": pred["mean"].to_numpy(),
        "ci_low": pred["mean_ci_lower"].to_numpy(),
        "ci_high": pred["mean_ci_upper"].to_numpy(),
        "analysis": label
    })

fig4_all = fit_predict_rcs_subset(analysis_df, "Full cohort")
fig4_gt1 = fit_predict_rcs_subset(analysis_df[analysis_df["los"] > 1], "Exclude LOS ≤1 day")
fig4_gt2 = fit_predict_rcs_subset(analysis_df[analysis_df["los"] > 2], "Exclude LOS ≤2 days")
FIG4_DATA = pd.concat([fig4_all, fig4_gt1, fig4_gt2], ignore_index=True)
FIG4_DATA.to_csv(OUT / "Figure4_Adjusted_RCS_Data.csv", index=False)

# Three panels, matching the manuscript description.
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
for ax, (label, g) in zip(axes, FIG4_DATA.groupby("analysis", sort=False)):
    ax.plot(g["LOS"], 100*g["predicted_probability"])
    ax.fill_between(g["LOS"], 100*g["ci_low"], 100*g["ci_high"], alpha=0.18)
    ax.set_title(label)
    ax.set_xlabel("Length of stay (days)")
    ax.grid(alpha=0.20)
axes[0].set_ylabel("Adjusted probability of in-hospital death (%)")
plt.tight_layout()
plt.savefig(OUT / "Figure4_Adjusted_RCS_Sensitivity.png", dpi=600, bbox_inches="tight")
plt.show()

LOS_SENSITIVITY_COUNTS = pd.DataFrame([
    {"analysis":"Full cohort with non-missing LOS", "N":analysis_df["los"].notna().sum(), "Deaths":analysis_df.loc[analysis_df["los"].notna(), TARGET].sum()},
    {"analysis":"LOS >1 day", "N":((analysis_df["los"]>1)&analysis_df["los"].notna()).sum(), "Deaths":analysis_df.loc[analysis_df["los"]>1, TARGET].sum()},
    {"analysis":"LOS >2 days", "N":((analysis_df["los"]>2)&analysis_df["los"].notna()).sum(), "Deaths":analysis_df.loc[analysis_df["los"]>2, TARGET].sum()},
])
LOS_SENSITIVITY_COUNTS["Mortality_pct"] = 100*LOS_SENSITIVITY_COUNTS["Deaths"]/LOS_SENSITIVITY_COUNTS["N"]
display(LOS_SENSITIVITY_COUNTS)
LOS_SENSITIVITY_COUNTS.to_csv(OUT / "LOS_Sensitivity_Counts.csv", index=False)

## Supplementary Table S5 and Supplementary Figure S1 — decision curve analysis from the SAME OOF predictions

In [ ]:
# CELL 13 — DCA: Supplementary Table S5 + Supplementary Figure S1

THRESHOLDS = np.arange(0.01, 0.101, 0.01)


def net_benefit(y_true, p_pred, thresholds):
    y_true = np.asarray(y_true).astype(int)
    p_pred = np.asarray(p_pred).astype(float)
    n = len(y_true)
    vals = []
    for pt in thresholds:
        treat = p_pred >= pt
        tp = np.sum(treat & (y_true == 1))
        fp = np.sum(treat & (y_true == 0))
        vals.append(tp/n - fp/n * (pt/(1-pt)))
    return np.asarray(vals)

prev = y.mean()
nbA = net_benefit(y, pA, THRESHOLDS)
nbB = net_benefit(y, pB, THRESHOLDS)
treat_all = prev - (1-prev)*(THRESHOLDS/(1-THRESHOLDS))
treat_none = np.zeros_like(THRESHOLDS)

S5 = pd.DataFrame({
    "threshold": THRESHOLDS,
    "net_benefit_ModelA": nbA,
    "net_benefit_ModelB": nbB,
    "net_benefit_TreatAll": treat_all,
    "net_benefit_TreatNone": treat_none
})
display(S5)
S5.to_csv(OUT / "Supplementary_Table_S5_Decision_Curve.csv", index=False)

useful_A = THRESHOLDS[(nbA > treat_all) & (nbA > treat_none)]
useful_B = THRESHOLDS[(nbB > treat_all) & (nbB > treat_none)]
print("Model A useful thresholds:", useful_A.tolist())
print("Model B useful thresholds:", useful_B.tolist())

plt.figure(figsize=(7.2, 5.2))
plt.plot(THRESHOLDS, nbA, label="Model A (admission-only)")
plt.plot(THRESHOLDS, nbB, label="Model B (inpatient-course)")
plt.plot(THRESHOLDS, treat_all, "--", label="Treat all")
plt.plot(THRESHOLDS, treat_none, "--", label="Treat none")
plt.xlabel("Threshold probability")
plt.ylabel("Net benefit")
plt.title("Decision curve analysis: patient-grouped OOF predictions")
plt.legend()
plt.tight_layout()
plt.savefig(OUT / "Supplementary_Figure_S1_Decision_Curve.png", dpi=600, bbox_inches="tight")
plt.show()

## Supplementary Table S6 — diagnosis-family patterns at LOS extremes

In [ ]:
# CELL 14 — Supplementary Table S6A/S6B/S6C
# Uses ONLY the locked G30.x cohort. This fixes the previous full-dataset contamination.
# Uses all diagnosis positions, deduplicates each 3-character family once per hospitalization,
# and excludes the defining G30 family and Z-code administrative/status families.

ICD_FAMILY_LABELS = {
    "E78":"Hyperlipidemia", "I10":"Hypertension", "E11":"Type 2 diabetes mellitus",
    "I25":"Chronic ischemic heart disease", "K21":"Gastroesophageal reflux disease",
    "I48":"Atrial fibrillation/flutter", "E87":"Fluid/electrolyte disorders",
    "I50":"Heart failure", "N18":"Chronic kidney disease", "J44":"COPD",
    "N17":"Acute kidney injury", "J96":"Respiratory failure", "A41":"Sepsis",
    "A40":"Streptococcal sepsis", "N39":"Urinary tract infection", "G93":"Other disorders of brain",
    "L89":"Pressure ulcer"
}


def family_prevalence(data):
    N = len(data)
    if N == 0:
        return pd.DataFrame(columns=["ICD10_family", "n_admissions", "percent", "label"])
    long = (data[DX_COLS].astype("string")
            .apply(lambda s: s.str.upper().str.strip())
            .stack(future_stack=True))
    long = long[long.notna() & (long != "")]
    code = long.str.replace(r"[^A-Z0-9]", "", regex=True).str.slice(0, 3)
    # Exclude cohort-defining AD family and administrative/status Z families.
    code = code[~code.str.startswith(("G30", "Z"), na=False)]
    tmp = code.reset_index()
    tmp.columns = ["row_id", "dx_position", "ICD10_family"]
    tmp = tmp.dropna(subset=["ICD10_family"]).drop_duplicates(["row_id", "ICD10_family"])
    counts = tmp["ICD10_family"].value_counts()
    out = counts.rename("n_admissions").reset_index()
    out["percent"] = 100*out["n_admissions"]/N
    out["label"] = out["ICD10_family"].map(ICD_FAMILY_LABELS).fillna("")
    return out

short_df = analysis_df[analysis_df["los"].notna() & (analysis_df["los"] <= 5)].copy()
long_df = analysis_df[analysis_df["los"].notna() & (analysis_df["los"] >= 25)].copy()

S6A = family_prevalence(short_df).head(25).copy()
S6A.insert(0, "LOS_group", "≤5 days")
S6A["N_group"] = len(short_df)

S6B = family_prevalence(long_df).head(25).copy()
S6B.insert(0, "LOS_group", "≥25 days")
S6B["N_group"] = len(long_df)

all_short = family_prevalence(short_df).rename(columns={"n_admissions":"n_short", "percent":"pct_short", "label":"label_short"})
all_long = family_prevalence(long_df).rename(columns={"n_admissions":"n_long", "percent":"pct_long", "label":"label_long"})
S6C = all_short.merge(all_long, on="ICD10_family", how="outer").fillna({"n_short":0,"pct_short":0,"n_long":0,"pct_long":0})
S6C["label"] = S6C["label_short"].replace("", np.nan).fillna(S6C["label_long"]).fillna("")
S6C["difference_pct_long_minus_short"] = S6C["pct_long"] - S6C["pct_short"]
S6C = S6C.sort_values("difference_pct_long_minus_short", ascending=False)

print("Short LOS N =", len(short_df), "| Prolonged LOS N =", len(long_df))
display(S6A)
display(S6B)
display(S6C.head(25))

S6A.to_csv(OUT / "Supplementary_Table_S6A_Short_LOS_Diagnosis_Families.csv", index=False)
S6B.to_csv(OUT / "Supplementary_Table_S6B_Prolonged_LOS_Diagnosis_Families.csv", index=False)
S6C.to_csv(OUT / "Supplementary_Table_S6C_LOS_Extreme_Comparison.csv", index=False)

# Pressure-ulcer context requested by Reviewer 3; timing/POA cannot be inferred here.
for name, fam in [("Short LOS ≤5", family_prevalence(short_df)), ("Prolonged LOS ≥25", family_prevalence(long_df))]:
    row = fam[fam["ICD10_family"] == "L89"]
    if row.empty:
        print(name, "L89 pressure ulcer: 0 detected")
    else:
        print(name, "L89 pressure ulcer:", int(row.iloc[0]["n_admissions"]), f"({row.iloc[0]['percent']:.2f}%)")

In [ ]:
# ============================================================
# FINAL CELL 14 — Supplementary Tables S6A, S6B, and S6C
# Diagnosis-family profiles at LOS extremes
# ============================================================
#
# REQUIREMENTS:
#   - analysis_df already exists and is the FINAL G30.x cohort
#   - DX_COLS already contains the ICD-10 diagnosis columns
#
# This cell:
#   1. Uses ONLY analysis_df
#   2. Defines short LOS as <=5 days
#   3. Defines prolonged LOS as >=25 days
#   4. Excludes G30/F01/F02/F03
#   5. Excludes Z and external-cause V/W/X/Y families
#   6. Counts each ICD family only once per hospitalization
#   7. Generates S6A, S6B, and S6C
#   8. Checks pressure ulcer (L89)
#   9. Verifies excluded codes are absent
#  10. Saves final CSV files
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 0. Safety checks
# ------------------------------------------------------------

if "analysis_df" not in globals():
    raise ValueError("analysis_df does not exist. Run the main analysis cells first.")

if "DX_COLS" not in globals():
    raise ValueError("DX_COLS does not exist. Run the diagnosis-column setup cell first.")

if "los" not in analysis_df.columns:
    raise ValueError("LOS variable 'los' is missing from analysis_df.")

# Use the existing output folder if already defined.
if "OUT" not in globals():
    OUT = Path("/content/nrd2017_ad_revision_outputs")

OUT.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# 1. ICD-10 family labels
# ------------------------------------------------------------

ICD_FAMILY_LABELS = {

    # Cardiovascular/metabolic
    "E78": "Hyperlipidemia",
    "I10": "Hypertension",
    "E11": "Type 2 diabetes mellitus",
    "I25": "Chronic ischemic heart disease",
    "I48": "Atrial fibrillation/flutter",
    "I50": "Heart failure",
    "I11": "Hypertensive heart disease",
    "I12": "Hypertensive chronic kidney disease",

    # Renal/electrolyte
    "N18": "Chronic kidney disease",
    "N17": "Acute kidney injury",
    "N39": "Urinary tract infection",
    "E87": "Fluid/electrolyte disorders",
    "E86": "Volume depletion",

    # Respiratory/infectious
    "J44": "Chronic obstructive pulmonary disease",
    "J96": "Respiratory failure",
    "J69": "Pneumonitis due to solids and liquids",
    "A41": "Sepsis",
    "A40": "Streptococcal sepsis",
    "B96": "Bacterial agents as cause of diseases classified elsewhere",
    "B37": "Candidiasis",

    # Neurologic/psychiatric
    "G93": "Other disorders of brain",
    "G47": "Sleep disorders",
    "F05": "Delirium due to known physiological condition",
    "F32": "Major depressive disorder, single episode",
    "F41": "Other anxiety disorders",
    "F22": "Delusional disorders",

    # Gastrointestinal/nutrition
    "K21": "Gastroesophageal reflux disease",
    "K59": "Other functional intestinal disorders",
    "R13": "Dysphagia",
    "E43": "Severe protein-calorie malnutrition",
    "E46": "Unspecified protein-calorie malnutrition",

    # Hematologic/endocrine
    "D64": "Other anemias",
    "E03": "Other hypothyroidism",
    "E83": "Disorders of mineral metabolism",
    "E55": "Vitamin D deficiency",

    # Other
    "L89": "Pressure ulcer",
    "M19": "Other and unspecified osteoarthritis",
    "N40": "Benign prostatic hyperplasia",
    "R45": "Symptoms involving emotional state",
    "R53": "Malaise and fatigue",
    "R62": "Lack of expected normal physiological development",
    "R65": "Systemic inflammatory response syndrome"
}


# ------------------------------------------------------------
# 2. ICD families to exclude
# ------------------------------------------------------------

# Cohort-defining / dementia-related families
EXCLUDE_EXACT_FAMILIES = {
    "G30",   # Alzheimer disease — cohort defining
    "F01",   # Vascular dementia
    "F02",   # Dementia in other diseases classified elsewhere
    "F03"    # Unspecified dementia
}

# Non-disease / external-cause families
EXCLUDE_FIRST_LETTERS = {
    "Z",     # Factors influencing health status / administrative
    "V",     # External causes
    "W",     # External causes
    "X",     # External causes
    "Y"      # External causes
}


# ------------------------------------------------------------
# 3. Function to calculate ICD-family prevalence
# ------------------------------------------------------------

def family_prevalence(data):

    N = len(data)

    if N == 0:
        return pd.DataFrame(
            columns=[
                "ICD10_family",
                "n_admissions",
                "percent",
                "label"
            ]
        )

    # Work from diagnosis columns only.
    dx = data[DX_COLS].copy()

    # Give every hospitalization a temporary unique row ID.
    dx.insert(
        0,
        "_row_id",
        np.arange(N)
    )

    # Convert wide diagnosis columns to long format.
    long = dx.melt(
        id_vars="_row_id",
        value_vars=DX_COLS,
        var_name="dx_position",
        value_name="code"
    )

    # Normalize ICD codes.
    long["code"] = (
        long["code"]
        .astype("string")
        .str.upper()
        .str.strip()
        .str.replace(
            r"[^A-Z0-9]",
            "",
            regex=True
        )
    )

    # Remove missing/blank values.
    long = long[
        long["code"].notna() &
        (long["code"] != "")
    ].copy()

    # Create 3-character ICD-10 family.
    long["ICD10_family"] = (
        long["code"]
        .str.slice(0, 3)
    )

    # Keep valid three-character families.
    long = long[
        long["ICD10_family"].str.len() == 3
    ].copy()

    # --------------------------------------------------------
    # Exclude Alzheimer/dementia families
    # --------------------------------------------------------

    long = long[
        ~long["ICD10_family"].isin(
            EXCLUDE_EXACT_FAMILIES
        )
    ].copy()

    # --------------------------------------------------------
    # Exclude Z and V/W/X/Y code families
    # --------------------------------------------------------

    long = long[
        ~long["ICD10_family"]
        .str[0]
        .isin(EXCLUDE_FIRST_LETTERS)
    ].copy()

    # --------------------------------------------------------
    # Count each ICD family only ONCE per hospitalization
    # --------------------------------------------------------

    long = long.drop_duplicates(
        subset=[
            "_row_id",
            "ICD10_family"
        ]
    )

    # Count hospitalizations containing each family.
    counts = (
        long["ICD10_family"]
        .value_counts()
        .rename("n_admissions")
        .reset_index()
    )

    counts.columns = [
        "ICD10_family",
        "n_admissions"
    ]

    # Calculate prevalence within LOS group.
    counts["percent"] = (
        100 *
        counts["n_admissions"] /
        N
    )

    # Add clinical labels.
    counts["label"] = (
        counts["ICD10_family"]
        .map(ICD_FAMILY_LABELS)
        .fillna("")
    )

    return counts


# ============================================================
# 4. Define LOS-extreme groups
# ============================================================

short_df = analysis_df[
    analysis_df["los"].notna() &
    (analysis_df["los"] <= 5)
].copy()

long_df = analysis_df[
    analysis_df["los"].notna() &
    (analysis_df["los"] >= 25)
].copy()


print("=" * 75)
print("LOS EXTREME GROUPS")
print("=" * 75)

print(
    f"Short LOS <=5 days: "
    f"N = {len(short_df):,}"
)

print(
    f"Prolonged LOS >=25 days: "
    f"N = {len(long_df):,}"
)


# ============================================================
# 5. Calculate all diagnosis-family prevalences
# ============================================================

short_all = family_prevalence(short_df)

long_all = family_prevalence(long_df)


# ============================================================
# 6. Supplementary Table S6A
# Top 25 diagnosis families in short LOS
# ============================================================

S6A = (
    short_all
    .head(25)
    .copy()
)

S6A.insert(
    0,
    "LOS_group",
    "<=5 days"
)

S6A["N_group"] = len(short_df)

# Round percentages for publication.
S6A["percent"] = (
    S6A["percent"]
    .round(2)
)


# ============================================================
# 7. Supplementary Table S6B
# Top 25 diagnosis families in prolonged LOS
# ============================================================

S6B = (
    long_all
    .head(25)
    .copy()
)

S6B.insert(
    0,
    "LOS_group",
    ">=25 days"
)

S6B["N_group"] = len(long_df)

S6B["percent"] = (
    S6B["percent"]
    .round(2)
)


# ============================================================
# 8. Supplementary Table S6C
# Direct short-vs-prolonged comparison
# ============================================================

short_compare = (
    short_all[
        [
            "ICD10_family",
            "n_admissions",
            "percent",
            "label"
        ]
    ]
    .rename(
        columns={
            "n_admissions": "n_short",
            "percent": "pct_short",
            "label": "label_short"
        }
    )
)

long_compare = (
    long_all[
        [
            "ICD10_family",
            "n_admissions",
            "percent",
            "label"
        ]
    ]
    .rename(
        columns={
            "n_admissions": "n_long",
            "percent": "pct_long",
            "label": "label_long"
        }
    )
)


S6C = short_compare.merge(
    long_compare,
    on="ICD10_family",
    how="outer"
)


# Missing family prevalence in either group = zero.
for c in [
    "n_short",
    "pct_short",
    "n_long",
    "pct_long"
]:
    S6C[c] = (
        S6C[c]
        .fillna(0)
    )


# Use whichever label is available.
S6C["label"] = (
    S6C["label_short"]
    .replace("", np.nan)
    .fillna(
        S6C["label_long"]
        .replace("", np.nan)
    )
    .fillna("")
)


# Calculate enrichment in prolonged LOS.
S6C[
    "difference_pct_long_minus_short"
] = (
    S6C["pct_long"] -
    S6C["pct_short"]
)


# Sort from strongest enrichment in prolonged LOS.
S6C = (
    S6C
    .sort_values(
        "difference_pct_long_minus_short",
        ascending=False
    )
    .reset_index(drop=True)
)


# Keep publication-friendly columns.
S6C = S6C[
    [
        "ICD10_family",
        "label",
        "n_short",
        "pct_short",
        "n_long",
        "pct_long",
        "difference_pct_long_minus_short"
    ]
]


# Round percentages.
for c in [
    "pct_short",
    "pct_long",
    "difference_pct_long_minus_short"
]:
    S6C[c] = (
        S6C[c]
        .round(2)
    )


# Convert counts back to integers.
S6C["n_short"] = (
    S6C["n_short"]
    .astype(int)
)

S6C["n_long"] = (
    S6C["n_long"]
    .astype(int)
)


# ============================================================
# 9. Display final S6A
# ============================================================

print("\n" + "=" * 75)
print("SUPPLEMENTARY TABLE S6A")
print("Top diagnosis families: LOS <=5 days")
print("=" * 75)

display(S6A)


# ============================================================
# 10. Display final S6B
# ============================================================

print("\n" + "=" * 75)
print("SUPPLEMENTARY TABLE S6B")
print("Top diagnosis families: LOS >=25 days")
print("=" * 75)

display(S6B)


# ============================================================
# 11. Display final S6C
# ============================================================

print("\n" + "=" * 75)
print("SUPPLEMENTARY TABLE S6C")
print("Diagnosis families most enriched in prolonged versus short LOS")
print("=" * 75)

display(
    S6C.head(25)
)


# ============================================================
# 12. Pressure-ulcer analysis for Reviewer 3
# ============================================================

print("\n" + "=" * 75)
print("PRESSURE ULCER (L89) CHECK")
print("=" * 75)


def get_family_result(table, family):

    row = table[
        table["ICD10_family"] == family
    ]

    if row.empty:
        return 0, 0.0

    return (
        int(row.iloc[0]["n_admissions"]),
        float(row.iloc[0]["percent"])
    )


short_l89_n, short_l89_pct = get_family_result(
    short_all,
    "L89"
)

long_l89_n, long_l89_pct = get_family_result(
    long_all,
    "L89"
)


print(
    f"Short LOS <=5 days: "
    f"{short_l89_n:,} "
    f"({short_l89_pct:.2f}%)"
)

print(
    f"Prolonged LOS >=25 days: "
    f"{long_l89_n:,} "
    f"({long_l89_pct:.2f}%)"
)

print(
    f"Absolute prevalence difference: "
    f"{long_l89_pct - short_l89_pct:.2f} percentage points"
)

if short_l89_pct > 0:
    print(
        f"Prevalence ratio "
        f"(prolonged / short): "
        f"{long_l89_pct / short_l89_pct:.2f}"
    )


# ============================================================
# 13. Verify excluded diagnosis families are gone
# ============================================================

print("\n" + "=" * 75)
print("EXCLUSION VALIDATION")
print("=" * 75)


def find_bad_codes(table):

    fam = (
        table["ICD10_family"]
        .astype("string")
    )

    bad = table[
        fam.isin(
            ["G30", "F01", "F02", "F03"]
        )
        |
        fam.str.startswith(
            ("V", "W", "X", "Y", "Z"),
            na=False
        )
    ]

    return bad


for name, table in [
    ("S6A", S6A),
    ("S6B", S6B),
    ("S6C", S6C)
]:

    bad = find_bad_codes(table)

    print(
        f"{name}: "
        f"excluded-code rows remaining = "
        f"{len(bad)}"
    )

    if len(bad) > 0:
        display(bad)


# Stop if an excluded family somehow remains.
all_bad = (
    len(find_bad_codes(S6A)) +
    len(find_bad_codes(S6B)) +
    len(find_bad_codes(S6C))
)

if all_bad > 0:
    raise ValueError(
        "Excluded ICD families remain in S6 tables. "
        "Do not use these outputs."
    )


# ============================================================
# 14. Check for blank labels among displayed top families
# ============================================================

print("\n" + "=" * 75)
print("LABEL CHECK")
print("=" * 75)


for name, table in [
    ("S6A", S6A),
    ("S6B", S6B),
    ("S6C top 25", S6C.head(25))
]:

    blank = table[
        table["label"]
        .astype("string")
        .str.strip()
        .eq("")
    ]

    print(
        f"{name}: blank labels = "
        f"{len(blank)}"
    )

    if len(blank) > 0:
        print(
            "Families without labels:",
            blank["ICD10_family"]
            .tolist()
        )


# ============================================================
# 15. Save FINAL supplementary tables
# ============================================================

file_s6a = (
    OUT /
    "Supplementary_Table_S6A_Short_LOS_Diagnosis_Families_FINAL.csv"
)

file_s6b = (
    OUT /
    "Supplementary_Table_S6B_Prolonged_LOS_Diagnosis_Families_FINAL.csv"
)

file_s6c = (
    OUT /
    "Supplementary_Table_S6C_LOS_Extreme_Comparison_FINAL.csv"
)


S6A.to_csv(
    file_s6a,
    index=False
)

S6B.to_csv(
    file_s6b,
    index=False
)

S6C.to_csv(
    file_s6c,
    index=False
)


print("\n" + "=" * 75)
print("FINAL S6 TABLES SAVED")
print("=" * 75)

print(file_s6a)
print(file_s6b)
print(file_s6c)

print("\nDONE.")

## Supplementary Table S7 — regularized logistic-regression comparator

In [ ]:
# CELL 15 — Supplementary Table S7


def grouped_logistic_oof(data, features, label):
    work = data[[TARGET, GROUP] + features].dropna(subset=[TARGET, GROUP]).reset_index(drop=True)
    X = work[features]
    y = work[TARGET].astype(int).to_numpy()
    groups = work[GROUP].astype(str).to_numpy()
    gkf = GroupKFold(n_splits=N_SPLITS)
    oof = np.full(len(work), np.nan)
    folds = []

    for fold, (tr, te) in enumerate(gkf.split(X, y, groups=groups), 1):
        Xtr, Xte = X.iloc[tr], X.iloc[te]
        ytr, yte = y[tr], y[te]
        pipe = make_logit_pipeline(features)
        pipe.fit(Xtr, ytr)
        p = pipe.predict_proba(Xte)[:, 1]
        oof[te] = p
        folds.append({"fold":fold, "AUROC":roc_auc_score(yte,p), "AUPRC":average_precision_score(yte,p)})

    f = pd.DataFrame(folds)
    return {
        "model": label, "N": len(y), "Deaths": int(y.sum()), "Prevalence": y.mean(),
        "pooled_OOF_AUROC": roc_auc_score(y,oof), "pooled_OOF_AUPRC": average_precision_score(y,oof),
        "fold_AUROC_mean": f.AUROC.mean(), "fold_AUROC_SD": f.AUROC.std(ddof=1),
        "fold_AUPRC_mean": f.AUPRC.mean(), "fold_AUPRC_SD": f.AUPRC.std(ddof=1)
    }

S7 = pd.DataFrame([
    grouped_logistic_oof(analysis_df, FEATURES_A, "Model A (admission-only)"),
    grouped_logistic_oof(analysis_df, FEATURES_B, "Model B (inpatient-course)")
])
display(S7)
S7.to_csv(OUT / "Supplementary_Table_S7_Logistic_Comparator.csv", index=False)

## Final audit, versions, and one ZIP containing all outputs

In [ ]:
# CELL 16 — Final consistency checks, software versions, output ZIP

import sklearn, statsmodels, xgboost, shap, patsy, scipy

# Key anti-contamination checks
assert "df" not in globals(), "A dataframe named df exists; restart runtime and run only this notebook."
assert "dat" not in globals(), "A dataframe named dat exists; restart runtime and run only this notebook."
assert "d" not in globals(), "A dataframe named d exists; restart runtime and run only this notebook."
assert analysis_df["has_g30"].all()
assert analysis_df["age"].ge(60).all()

versions = pd.DataFrame({
    "software": ["Python", "pandas", "numpy", "scipy", "statsmodels", "scikit-learn", "xgboost", "shap", "patsy"],
    "version": [sys.version.split()[0], pd.__version__, np.__version__, scipy.__version__, statsmodels.__version__,
                sklearn.__version__, xgboost.__version__, shap.__version__, patsy.__version__]
})
versions.to_csv(OUT / "Software_Versions.csv", index=False)
display(versions)

# Save compact results summary for manuscript updating.
summary = {
    "analytic_N": int(len(analysis_df)),
    "deaths": int(analysis_df[TARGET].sum()),
    "unweighted_mortality_pct": float(100*analysis_df[TARGET].mean()),
    "weighted_N": float(analysis_df["discwt"].sum()),
    "weighted_deaths": float((analysis_df["discwt"]*analysis_df[TARGET]).sum()),
    "weighted_mortality_pct": float(100*np.average(analysis_df[TARGET], weights=analysis_df["discwt"])),
    "missing_LOS": int(analysis_df["los"].isna().sum()),
    "ModelA_pooled_OOF_AUROC": float(ML_A["pooled"]["pooled_OOF_AUROC"]),
    "ModelA_pooled_OOF_AUPRC": float(ML_A["pooled"]["pooled_OOF_AUPRC"]),
    "ModelB_pooled_OOF_AUROC": float(ML_B["pooled"]["pooled_OOF_AUROC"]),
    "ModelB_pooled_OOF_AUPRC": float(ML_B["pooled"]["pooled_OOF_AUPRC"]),
    "LOS_spline_Wald_p": LOS_SPLINE_WALD_P,
    "LOS_RCS_vs_linear_LRT_p": LOS_NONLINEAR_LRT_P,
    "short_LOS_le5_N": int(len(short_df)),
    "prolonged_LOS_ge25_N": int(len(long_df))
}
with open(OUT / "Manuscript_Update_Summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))

zip_path = Path("/content/NRD_2017_AD_G30_revision_outputs.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.glob("*")):
        z.write(p, arcname=p.name)

print("\nAll outputs saved to:", OUT)
print("ZIP:", zip_path)
files.download(str(zip_path))